### Import needed libraries

In [1]:
import os
import pickle
import numpy as np
from sklearn.model_selection import train_test_split # type: ignore
from sklearn.preprocessing import LabelEncoder # type: ignore
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Conv1D, Dense, Flatten, BatchNormalization, Dropout # type: ignore
from tensorflow.keras.utils import to_categorical # type: ignore

import os

# Point XLA to your CUDA/libdevice directory
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/lib/cuda'


2025-09-17 16:57:31.884179: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Load data

In [2]:
pickle_files = ["good.pickle", "hello.pickle", "nosign.pickle", "afternoon.pickle"]
X = []
y = []

for file in pickle_files:
    if os.path.exists(file):
        with open(file, "rb") as f:
            data = pickle.load(f)
        for entry in data:
            points = entry["points"]  # points is a list of [x, y]
            if points:  # skip empty
                X.append(points)
                y.append(entry["class_name"])
    else:
        print(f"{file} not found!")

max_points = max(len(sample) for sample in X)

X_padded = []
for sample in X:
    arr = np.array(sample, dtype=np.float32)
    if len(arr) < max_points:
        padding = np.zeros((max_points - len(arr), 2), dtype=np.float32)
        arr = np.vstack([arr, padding])
    X_padded.append(arr)

In [3]:
print(f"Total samples: {len(X_padded)}")
print(f"Max points: {max_points}")
print(f"Min points: {min(len(sample) for sample in X_padded)}")
print(f"Average points: {sum(len(sample) for sample in X_padded) / len(X_padded)}")
print(f"Unique classes: {len(set(y))}")
print(f"Class distribution: {dict(zip(set(y), [y.count(class_name) for class_name in set(y)]))}")
print(f"Sample shape: {X_padded[0].shape}")
print(f"Sample: {X_padded[0]}")
print(f"Label: {y[0]}")

Total samples: 1620
Max points: 213
Min points: 213
Average points: 213.0
Unique classes: 4
Class distribution: {'afternoon': 360, 'good': 500, 'nosign': 320, 'hello': 440}
Sample shape: (213, 2)
Sample: [[306. 204.]
 [309. 206.]
 [309. 206.]
 [312. 208.]
 [312. 208.]
 [317. 211.]
 [317. 211.]
 [323. 213.]
 [323. 213.]
 [331. 214.]
 [331. 214.]
 [338. 214.]
 [338. 214.]
 [344. 212.]
 [344. 212.]
 [350. 209.]
 [350. 209.]
 [354. 206.]
 [354. 206.]
 [356. 204.]
 [306. 204.]
 [308. 202.]
 [308. 202.]
 [311. 200.]
 [311. 200.]
 [315. 198.]
 [315. 198.]
 [323. 196.]
 [323. 196.]
 [330. 197.]
 [330. 197.]
 [338. 196.]
 [338. 196.]
 [346. 198.]
 [346. 198.]
 [351. 200.]
 [351. 200.]
 [355. 202.]
 [355. 202.]
 [356. 204.]
 [310. 203.]
 [313. 204.]
 [313. 204.]
 [315. 204.]
 [315. 204.]
 [320. 204.]
 [320. 204.]
 [325. 205.]
 [325. 205.]
 [331. 205.]
 [331. 205.]
 [337. 205.]
 [337. 205.]
 [342. 204.]
 [342. 204.]
 [347. 204.]
 [347. 204.]
 [350. 204.]
 [350. 204.]
 [353. 203.]
 [310. 203.]
 [3

In [4]:
X_array = np.array(X_padded)
X_array = X_array.reshape(X_array.shape[0], X_array.shape[1], 2)

print(f"Input shape: {X_array.shape}")

le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_categorical = to_categorical(y_encoded)

X_train, X_test, y_train, y_test = train_test_split(
    X_array, y_categorical, test_size=0.2, random_state=42, stratify=y_categorical
)

print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

model = Sequential([
    Conv1D(64, kernel_size=3, activation='relu', input_shape=(X_array.shape[1], 2)),
    Conv1D(64, kernel_size=3, activation='relu'),
    Dropout(0.3),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(len(le.classes_), activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=150,
    batch_size=32,
    verbose=2
)

loss, acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {acc*100:.2f}%")

model.save("gesture_recognition_cnn.h5", save_format="h5")

print("Model saved as gesture_recognition_cnn.h5")

Input shape: (1620, 213, 2)
Training samples: 1296, Test samples: 324


/home/yassin/miniconda3/envs/tf/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1758121055.540490  136448 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5356 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 211, 64)        │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 209, 64)        │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 209, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 13376)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,712,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,725,572 (6.58 MB)

...

 Non-trainable params: 0 (0.00 B)

Epoch 1/150


2025-09-17 16:57:37.499419: I external/local_xla/xla/service/service.cc:163] XLA service 0x764c0c00a770 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-09-17 16:57:37.499432: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3060 Ti, Compute Capability 8.6
2025-09-17 16:57:37.544153: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-09-17 16:57:37.766945: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002
2025-09-17 16:57:37.893754: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-09-17 16:57:38.914299: 

41/41 - 10s - 236ms/step - accuracy: 0.3711 - loss: 29.8534 - val_accuracy: 0.3642 - val_loss: 1.0973
Epoch 2/150
41/41 - 0s - 4ms/step - accuracy: 0.4475 - loss: 1.1836 - val_accuracy: 0.5463 - val_loss: 1.0465
Epoch 3/150
41/41 - 0s - 4ms/step - accuracy: 0.4738 - loss: 1.0501 - val_accuracy: 0.5586 - val_loss: 0.9738
Epoch 4/150
41/41 - 0s - 4ms/step - accuracy: 0.5077 - loss: 1.0220 - val_accuracy: 0.5340 - val_loss: 0.8938
Epoch 5/150
41/41 - 0s - 4ms/step - accuracy: 0.5440 - loss: 0.9609 - val_accuracy: 0.6049 - val_loss: 0.8800
Epoch 6/150
41/41 - 0s - 4ms/step - accuracy: 0.5363 - loss: 0.9339 - val_accuracy: 0.6019 - val_loss: 0.8842
Epoch 7/150
41/41 - 0s - 4ms/step - accuracy: 0.5741 - loss: 0.8996 - val_accuracy: 0.6235 - val_loss: 0.8086
Epoch 8/150
41/41 - 0s - 4ms/step - accuracy: 0.6073 - loss: 0.8452 - val_accuracy: 0.7099 - val_loss: 0.7478
Epoch 9/150
41/41 - 0s - 4ms/step - accuracy: 0.6543 - loss: 0.7800 - val_accuracy: 0.7901 - val_loss: 0.6243
Epoch 10/150
41/41

Test accuracy: 82.72%
Model saved as gesture_recognition_cnn.h5
